In [0]:
%run ../init/kafka_config

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import col, to_json, struct, udf
import uuid
import random

In [0]:
generate_uuid   = udf(lambda: str(uuid.uuid4()),         StringType())
generate_item   = udf(lambda: random.choice(
                    ["I001","I002","I003","I004","I005"]),StringType())
generate_qty    = udf(lambda: random.randint(1, 5),      IntegerType())

In [0]:
user_pool = [
    ("U001", "normal"),
    ("U002", "normal"),
    ("U003", "normal"),
    ("U004", "premium"),
    ("U005", "normal"),
]


In [0]:
num_orders = 10

# randomly pick users for each order
import random
orders_raw = [random.choice(user_pool) for _ in range(num_orders)]

# create base DataFrame with user_id and category
orders_schema = StructType([
    StructField("user_id",  StringType(), nullable=False),
    StructField("category", StringType(), nullable=False)
])

orders_df = spark.createDataFrame(orders_raw, orders_schema)

# add order_id, item_id, item_quantity
orders_df = orders_df \
    .withColumn("order_id",      generate_uuid()) \
    .withColumn("item_id",       generate_item()) \
    .withColumn("item_quantity", generate_qty())

# reorder columns cleanly
orders_df = orders_df.select(
    "order_id",
    "user_id",
    "item_id",
    "item_quantity",
    "category"
)

print("── Generated Orders ────────────────────")
orders_df.show(truncate=False)


In [0]:
# Kafka partition routing:
# normal  → partition 0 or 1 (randomly)
# premium → partition 2

from pyspark.sql.functions import when, rand, floor

orders_df = orders_df.withColumn("partition",
    when(col("category") == "premium",
        2)                                      # premium always → partition 2
    .otherwise(
        (floor(rand() * 2)).cast(IntegerType()) # normal → 0 or 1 randomly
    )
)

In [0]:
# Kafka requires:
# value column → message body (JSON string as bytes)
# key column   → routing key (optional but good practice)

kafka_df = orders_df \
    .withColumn("value",
        to_json(struct(
            col("order_id"),
            col("user_id"),
            col("item_id"),
            col("item_quantity"),
            col("category")
        ))
    ) \
    .withColumn("key", col("category")) \
    .select("key","value","partition")

print("── Kafka DataFrame ─────────────────────")
kafka_df.show(truncate=False)

In [0]:
kafka_df.write \
    .format("kafka") \
    .option("kafka.bootstrap.servers", bootstrap) \
    .option("kafka.security.protocol", "SASL_SSL") \
    .option("kafka.sasl.mechanism", "PLAIN") \
    .option("kafka.sasl.jaas.config",
        f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";') \
    .option("topic", "orders") \
    .save()

print(f"\n✓ {num_orders} orders sent to Kafka successfully")
print("── Partition Summary ───────────────────")
orders_df.groupBy("category","partition").count().show()